# Anonymous review adaptation

This notebook is adapted from the research implementation. Model labels and output names use **Adjustment**. Personal paths are replaced by a local configuration. Hyperparameters, data screens, and estimation logic are otherwise retained. The MiniLM benchmark uses the same external-snapshot alignment as the other encoders.

**Required:** a locally authorized analysis sample and completed embedding export. These inputs are not bundled. This adaptation has passed static checks and CPU component tests; full neural training has not been rerun in the packaging environment. New code identities create separate checkpoints; original results remain unchanged.


In [ ]:
from pathlib import Path
import runpy
_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p/'src'/'project_config.py').is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks folder.")
_config_api = runpy.run_path(str(REPO_ROOT/'src'/'project_config.py'))
REPO_CONFIG = _config_api['load_project_config'](REPO_ROOT)
print('Repository configuration loaded. Input files are read locally; nothing is uploaded.')


# 10-K competition Adjustment: Llama semi-synthetic Monte Carlo (50 repetitions)

**Question:** When the causal slope is known, does the current adversarial representation reduce estimation error relative to W-only, Raw H and the same-budget NoAdv representation? The result is not predetermined. The design is inspired by your uploaded `Adjustment_simulation_0728(3).ipynb` and its C/M/L decomposition. It uses the current 10-K Adjustment implementation rather than importing its earlier Upworthy-specific row folds or tuned candidate search.

Real: Llama report embeddings, four financial controls, fiscal-year/industry categories, document/company IDs. Synthetic: treatment T, outcome Y and known semantic-factor roles. The observed competition variable is used to validate text/vector alignment, **not as the simulated treatment**. Actual ROA is not used as Y. Projection factors are artificial semantic coordinates, not verified measures of market competition.

Default: four scenarios × **50 replications per scenario** = 200 tasks; 1,000 distinct companies per replication; one real report per company. This keeps the first test small and avoids panel errors requiring additional dependence modeling. It does not evaluate a full repeated-firm panel estimator. Every method shares the same sampled reports, T/Y draws and outer folds within each task. The same draws are paired across scenarios.

Known potential outcomes: `Y(t) = tau*t + g(C,M,W) + epsilon_Y`. Default `tau=1.0`. Thus `Y(t+1)-Y(t)=1` for every report. This is a constant unit-increase effect in **synthetic units**, not a claim that one real competition percentage point changes ROA by 100 percentage points.


## Design locked before looking at method performance

| Scenario | C coefficient in T | T-only L coefficient in T | Role |
|---|---:|---:|---|
| Low confounding | 0.15 | 1.0 | Weak omitted-confounding stress |
| Medium confounding | 0.60 | 1.0 | Moderate stress |
| High confounding | 1.20 | 1.0 | Strong stress |
| High confounding, no T-only signal | 1.20 | 0.0 | Boundary: removing T information may destroy confounders |

`T = gamma_C*C + gamma_L*L + 0.3*W_score + epsilon_T`

`Y = tau*T + 1.0*C + 0.5*M + 0.3*W_score + epsilon_Y`

Independent Gaussian treatment/outcome errors have SD 1. Neither T nor Y is re-standardized after generation. Conditional treatment noise variance remains positive and fixed as confounding changes. Nonlinear C/M and simpler L come from disjoint PCA-coordinate blocks of real Llama H; calibration residualization reduces linear association but does not imply independence. Scenario names refer to structural coefficients; realized correlations are exported for checking.

**Terminology:** L is a pre-treatment, treatment-only / instrument-like signal, not a descendant of T or proof of post-treatment leakage. Raw H is valid pre-treatment information in this specified DGP. Adjustment has no guaranteed advantage. A lower R²(T from Z) can result from losing the confounder C; C/M/L recovery probes explicitly check that risk. This design does not fabricate post-treatment H or force Raw H to be invalid.

Ten methods: eight W-only/Raw-H/NoAdv/Adjustment × Ridge/RF, plus learned Oracle-components Ridge and exact Oracle. Exact Oracle knows both true nuisance functions and checks DGP/score implementation. Oracle factors never enter the eight ordinary methods or representation early stopping. There is no method selection using the true effect.


## Step 1 — Settings and imports
Uses the moved Windows folders. Requires the completed Llama report export and the already aligned financial analysis table. No new embeddings, SEC requests or paid API calls. Use your current PyTorch environment; if `threadpoolctl` or `tqdm` is missing, install only the missing package.

Compute-saving differences from the empirical run: 2 rather than 5 folds; 20 rather than 40 maximum epochs; 8 rather than 10 warm-up epochs; batch size 256; 60 rather than 300 RF trees, depth 8, leaf minimum 10. The architecture (Z=64), adversarial strengths and Ridge penalty remain unchanged. All eight benchmarks receive the same budget. Cholesky Ridge solves the same penalized objective with less repeated SVD cost. This is a development simulation, not final publication-level evidence.

Numeric W is clipped at calibration-company 1%/99% thresholds **for this synthetic design only** to prevent the previously audited extreme current ROA from dominating DGP scale; the same transformed W goes to every method. This does not modify your empirical data or select samples by future Y. Year and SIC2 remain categorical controls.

Do not run this alongside Llama embedding inference on a memory-limited GPU. For a quick end-to-end smoke test, use `replications=2` before launching a new locked 50-replication run. Changed settings generate a different output fingerprint.


In [ ]:
from pathlib import Path
import os,json,time,hashlib,random,gc,copy
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from scipy.stats import t as student_t
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold,GroupShuffleSplit
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from IPython.display import display
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset,DataLoader

PROJECT_ROOT=REPO_ROOT
COMPUSTAT_ROOT=REPO_CONFIG['data_root']
EMBEDDING_ROOT=REPO_CONFIG['embedding_root']
DATA_DIR=REPO_CONFIG['analysis_data_dir']
OUTPUT_ROOT=REPO_CONFIG['output_root']/'semisynthetic_llama'
EMBEDDING_ENGINE='llama'
EXPECTED_CORPUS_ID=REPO_CONFIG['expected_corpus_id']
EMBEDDING_SNAPSHOT=REPO_CONFIG['embedding_snapshots'].get('llama')
NUM_COLS=['w_roa_current','w_log_assets','w_leverage','w_cash_ratio']
CAT_COLS=['fe_year','fe_industry_sic2']
SIM={'version':'llama_semisynthetic_v1','replications':50,'n_companies':1000,
     'calibration_companies':1500,'calibration_seed':9031,'base_seed':85001,
     'pca_components':12,'true_taus':[1.0],'sigma_t':1.0,'sigma_y':1.0,
     'beta_c':1.0,'beta_m':0.5,'beta_w':0.3,'gamma_w':0.3,
     'clip_numeric_quantiles':[0.01,0.99]}
SCENARIOS={
 'low_confounding':{'gamma_c':0.15,'gamma_l':1.0},
 'medium_confounding':{'gamma_c':0.60,'gamma_l':1.0},
 'high_confounding':{'gamma_c':1.20,'gamma_l':1.0},
 'high_confounding_no_T_only':{'gamma_c':1.20,'gamma_l':0.0},
}
# Same architecture and adversarial strength as the empirical 10-K notebook.
# Smaller compute budget, applied equally to NoAdv and Adjustment. No tuning against truth.
CFG={'seed':42,'n_splits':2,'dZ':64,'encoder_hidden':[256,128],
 'outcome_hidden':[128,64],'adversary_hidden':[64,32],
 'dropout':0.1,'grl_lambda':0.05,'adv_weight':0.005,'lr':0.001,
 'weight_decay':1e-5,'epochs':20,'min_epochs':8,'patience':4,'batch_size':256,
 'ridge_alpha':10.0,'rf_n_estimators':60,'rf_max_depth':8,
 'rf_min_samples_leaf':10,'rf_max_features':0.5}
CPU_THREADS=min(4,max(1,(os.cpu_count() or 2)-1))
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(CPU_THREADS)
assert SIM['replications']>=2 and SIM['n_companies']>=100
assert CFG['epochs']>=CFG['min_epochs']
print('Device:',DEVICE,'| companies per replication:',SIM['n_companies'])
print('Scenario x replication tasks:',len(SCENARIOS)*SIM['replications']*len(SIM['true_taus']))
print('Two folds; two learned representations per fold; eight non-oracle methods.')

def set_seed(seed):
 random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def atom_json(path,value):
 path=Path(path);temp=path.with_name(path.name+'.tmp')
 temp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8');os.replace(temp,path)

def atomic_npz(path,**arrays):
 temp=path.with_name(path.name+'.tmp')
 with temp.open('wb') as f:np.savez_compressed(f,**arrays)
 os.replace(temp,path)

from threadpoolctl import threadpool_limits
# Avoid excessive BLAS threading in repeated small regressions.
_blas_limit=threadpool_limits(limits=CPU_THREADS)


## Step 2 — File alignment, synthetic design and exact-score utilities
The completed snapshot is checked by checksum and corpus ID. Report vectors join by doc_id. Missing embeddings stop the run. DGP construction uses 1,500 held-apart companies; they never appear in simulation evaluation samples. Calibration PCA/scalers/factor functions use no generated treatments/outcomes and are fixed thereafter. Model preprocessing fits its own outer or inner training data only.


In [ ]:
def embedding_sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()

def resolve_embedding():
 if EMBEDDING_SNAPSHOT is not None:
  choices=[Path(EMBEDDING_SNAPSHOT)]
 else:
  choices=[]
  for path in (EMBEDDING_ROOT/EMBEDDING_ENGINE/EXPECTED_CORPUS_ID).rglob('latest_snapshot.json'):
   candidate=json.loads(path.read_text(encoding='utf-8'))
   if candidate.get('complete') is True and candidate.get('corpus_id')==EXPECTED_CORPUS_ID:
    choices.append(path)
 if len(choices)!=1:
  raise RuntimeError(f'Expected one completed {EMBEDDING_ENGINE} snapshot; found {len(choices)}: {choices}. Finish export or set EMBEDDING_SNAPSHOT explicitly.')
 path=choices[0];snapshot=json.loads(path.read_text(encoding='utf-8'))
 assert snapshot.get('complete') is True, 'Embedding export is incomplete.'
 assert snapshot['corpus_id']==EXPECTED_CORPUS_ID, 'Different text corpus.'
 matrix=path.parent/snapshot['report_matrix'];index=path.parent/snapshot['report_index']
 assert embedding_sha(matrix)==snapshot['report_matrix_sha256'], 'Embedding matrix checksum mismatch.'
 assert embedding_sha(index)==snapshot['report_index_sha256'], 'Embedding index checksum mismatch.'
 return path,snapshot,matrix,index

_embedding_cache={}
def aligned_report_vectors(source):
 if 'matrix' not in _embedding_cache:
  idx=pd.read_csv(SELECTED_INDEX,dtype={'doc_id':'string','corpus_id':'string'})
  matrix=np.load(SELECTED_MATRIX,mmap_mode='r',allow_pickle=False)
  assert idx.doc_id.notna().all() and idx.doc_id.is_unique, 'Missing/duplicate embedding doc_id.'
  rows=pd.to_numeric(idx.embedding_row,errors='raise').to_numpy(dtype=float)
  assert np.isfinite(rows).all() and np.equal(rows,np.floor(rows)).all()
  assert np.array_equal(np.sort(rows),np.arange(len(idx))), 'Invalid embedding row index.'
  assert matrix.shape==(len(idx),int(SELECTED_SNAPSHOT['dimension']))
  assert len(idx)==int(SELECTED_SNAPSHOT['reports_exported'])
  assert idx.corpus_id.eq(EXPECTED_CORPUS_ID).all()
  _embedding_cache.update(index=idx.set_index('doc_id'),matrix=matrix)
 idx=_embedding_cache['index'];matrix=_embedding_cache['matrix']
 missing=source.loc[~source.doc_id.isin(idx.index),'doc_id']
 assert missing.empty, f'Missing {len(missing)} report embeddings. No silent sample deletion is allowed.'
 matched=idx.loc[source.doc_id]
 assert np.allclose(pd.to_numeric(matched.competition_intensity),pd.to_numeric(source.competition_intensity),rtol=1e-8,atol=1e-10), 'Competition text corpus mismatch.'
 aligned=np.asarray(matrix[matched.embedding_row.to_numpy(dtype=np.int64)],dtype=np.float32)
 assert np.isfinite(aligned).all() and (np.linalg.norm(aligned,axis=1)>0).all()
 return aligned

def one_report_per_company(frame,seed):
 rng=np.random.default_rng(seed)
 # Sampling every company equally avoids report-rich companies dominating the design.
 return np.array([rng.choice(indices) for indices in frame.groupby('gvkey',sort=True).indices.values()],dtype=int)

def raw_semantic_factors(u):
 relu=lambda x:np.maximum(x,0)
 return np.column_stack([
  0.60*relu(u[:,0])-0.45*relu(-u[:,1])+0.40*np.tanh(u[:,2])+0.25*relu(u[:,0]+u[:,3]),
  0.60*np.tanh(u[:,4])-0.40*relu(-u[:,5])+0.30*relu(u[:,6]+u[:,7]),
  0.80*u[:,8]+0.45*u[:,9]+0.20*np.tanh(u[:,10])+0.15*u[:,11]])

class FactorDesign:
 """Fit the synthetic DGP on held-apart calibration companies only; no T or Y is used."""
 def fit(self,h,wn,wc):
  lo,hi=SIM['clip_numeric_quantiles']
  self.lo=np.quantile(wn,lo,axis=0);self.hi=np.quantile(wn,hi,axis=0)
  self.ws=StandardScaler().fit(np.clip(wn,self.lo,self.hi))
  self.cat=OneHotEncoder(handle_unknown='ignore',sparse_output=False).fit(wc)
  w=self.controls(wn,wc)
  rng=np.random.default_rng(SIM['calibration_seed'])
  self.wcoef=rng.normal(size=w.shape[1]);self.wcoef/=np.linalg.norm(self.wcoef)
  ws=w@self.wcoef;self.wmean=ws.mean();self.wsd=ws.std()
  self.hs=StandardScaler().fit(h)
  self.pca=PCA(n_components=SIM['pca_components'],whiten=True,svd_solver='randomized',random_state=SIM['calibration_seed'])
  raw=raw_semantic_factors(self.pca.fit_transform(self.hs.transform(h)))
  self.models=[];self.means=[];self.sds=[];previous=[]
  for k in range(3):
   x=np.column_stack([w,*previous]) if previous else w
   model=make_pipeline(StandardScaler(),Ridge(alpha=10.,solver='svd')).fit(x,raw[:,k])
   r=raw[:,k]-model.predict(x);mu=r.mean();sd=r.std()
   assert sd>1e-8
   previous.append((r-mu)/sd)
   self.models.append(model);self.means.append(mu);self.sds.append(sd)
  return self
 def controls(self,wn,wc):
  return np.column_stack([self.ws.transform(np.clip(wn,self.lo,self.hi)),self.cat.transform(wc)])
 def transform(self,h,wn,wc):
  w=self.controls(wn,wc);raw=raw_semantic_factors(self.pca.transform(self.hs.transform(h)))
  previous=[]
  for k in range(3):
   x=np.column_stack([w,*previous]) if previous else w
   previous.append((raw[:,k]-self.models[k].predict(x)-self.means[k])/self.sds[k])
  c,m,l=previous
  score=(w@self.wcoef-self.wmean)/self.wsd
  return {'C':c,'M':m,'L':l,'W_score':score,'W_numeric':np.clip(wn,self.lo,self.hi)}

def generate_dgp(factors,params,tau,seed):
 rng=np.random.default_rng(seed)
 c,m,l,w=[factors[k] for k in ['C','M','L','W_score']]
 # Do not re-standardize T or Y after generation: tau remains exactly in these units.
 m_t=params['gamma_c']*c+params['gamma_l']*l+SIM['gamma_w']*w
 g_y=SIM['beta_c']*c+SIM['beta_m']*m+SIM['beta_w']*w
 t=m_t+rng.normal(0,SIM['sigma_t'],len(c))
 eps=rng.normal(0,SIM['sigma_y'],len(c))
 y=tau*t+g_y+eps
 # Exact oracle: E[T|C,M,L,W]=m_t; E[Y|C,M,L,W]=tau*m_t+g_y.
 return {'T':t,'Y':y,'m_true':m_t,'g_true':g_y,'ell_true':tau*m_t+g_y,'epsilon_y':eps}

def score_summary(t,y,that,yhat,groups,tau):
 u=t-that;v=y-yhat;denom=np.mean(u*u)
 assert denom>1e-10
 theta=np.mean(u*v)/denom;psi=u*(v-theta*u)/denom
 sums=pd.DataFrame({'group':groups,'psi':psi}).groupby('group').psi.sum().to_numpy()
 n=len(t);g=len(sums)
 se=np.sqrt(g/(g-1)*np.sum(sums*sums)/(n*n));crit=student_t.ppf(.975,g-1)
 lo,hi=theta-crit*se,theta+crit*se
 return {'estimate':float(theta),'se':float(se),'ci_low':float(lo),'ci_high':float(hi),
  'covered':bool(lo<=tau<=hi),'error':float(theta-tau),'absolute_error':float(abs(theta-tau)),
  'squared_error':float((theta-tau)**2),'R2_T':r2_np(t,that),'R2_Y':r2_np(y,yhat),
  'residual_mse_to_T_variance':float(denom/np.var(t)),'N':n,'companies':g}


## Step 3 — Load the shared real design and freeze the experiment
This reads report vectors only, not millions of paragraph vectors. The original empirical table is read only. Configuration, input hashes, package versions, code digest and embedding settings determine a new results folder.


In [ ]:
CODE_DIGEST='af623643af1652e38fb06d1798a62d0f361125237bb7e253ffc44a8ae8ffb47d'
SELECTED_SNAPSHOT_PATH,SELECTED_SNAPSHOT,SELECTED_MATRIX,SELECTED_INDEX=resolve_embedding()
receipt=json.loads((DATA_DIR/'dataset_snapshot.json').read_text(encoding='utf-8'))
assert receipt['complete']
table_path=DATA_DIR/receipt['table']
assert embedding_sha(table_path)==receipt['table_sha256']
source=pd.read_csv(table_path,dtype={'doc_id':'string','gvkey':'string',**{c:'string' for c in CAT_COLS}},low_memory=False)
assert source.doc_id.is_unique and source.doc_id.notna().all() and source.gvkey.notna().all()
assert source[NUM_COLS+CAT_COLS].notna().all().all()
source_H=aligned_report_vectors(source)
assert np.isfinite(source[NUM_COLS].to_numpy(dtype=float)).all()
company_ids=np.sort(source.gvkey.unique());rng=np.random.default_rng(SIM['calibration_seed'])
assert len(company_ids)>SIM['calibration_companies']+SIM['n_companies']
calibration_ids=rng.choice(company_ids,SIM['calibration_companies'],replace=False)
cal_frame=source.loc[source.gvkey.isin(calibration_ids)].reset_index().rename(columns={'index':'source_row'})
cal_rows=cal_frame.iloc[one_report_per_company(cal_frame,SIM['calibration_seed'])].source_row.to_numpy(dtype=int)
evaluation_rows=np.flatnonzero(~source.gvkey.isin(calibration_ids).to_numpy())
evaluation_frame=source.iloc[evaluation_rows].reset_index(drop=True)
assert set(source.iloc[cal_rows].gvkey).isdisjoint(set(evaluation_frame.gvkey))
wn=source[NUM_COLS].to_numpy(dtype=float);wc=source[CAT_COLS].astype(str).to_numpy()
design=FactorDesign().fit(source_H[cal_rows],wn[cal_rows],wc[cal_rows])
all_factors=design.transform(source_H,wn,wc)
# Real covariates are fixed design inputs; all generated T/Y are fresh per replication.
versions={k:metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','torch']}
run_identity={'simulation':SIM,'scenarios':SCENARIOS,'model':CFG,'code':CODE_DIGEST,
 'table_sha256':receipt['table_sha256'],'embedding_matrix_sha256':SELECTED_SNAPSHOT['report_matrix_sha256'],
 'embedding_index_sha256':SELECTED_SNAPSHOT['report_index_sha256'],
 'embedding_settings':SELECTED_SNAPSHOT['settings'],'versions':versions,'device':DEVICE,'cpu_threads':CPU_THREADS}
RUN_ID=hashlib.sha256(json.dumps(run_identity,sort_keys=True).encode()).hexdigest()[:20]
RUN_DIR=OUTPUT_ROOT/RUN_ID;RUN_DIR.mkdir(parents=True,exist_ok=True)
atom_json(RUN_DIR/'locked_configuration.json',run_identity)
joblib.dump(design,RUN_DIR/'dgp_calibration.joblib')
source[['doc_id','gvkey']].assign(calibration_company=source.gvkey.isin(calibration_ids).to_numpy()).to_csv(RUN_DIR/'design_population.csv.gz',index=False,compression='gzip')
pd.DataFrame({k:all_factors[k][evaluation_rows] for k in ['C','M','L','W_score']}).corr().to_csv(RUN_DIR/'factor_correlations.csv')
print('Original reports:',len(source),'| embedding:',source_H.shape)
print('Calibration companies:',len(calibration_ids),'| evaluation companies:',evaluation_frame.gvkey.nunique())
print('Output:',RUN_DIR)


## Step 4 — Current Adjustment model and nuisance learners
The same H→Z encoder, outcome head [Z,W,T], adversary from Z, alternating updates and gradient reversal are used. NoAdv zeros adversarial pressure. Both select minimum inner validation outcome MSE after warm-up, as in the current 10-K empirical notebook. The older uploaded Python files used a different outcome-tolerance/treatment-probe checkpoint rule; that rule is **not silently substituted here**. Both learned views share their Z across Ridge and RF. Outcome nuisances predict Y without T; the training Y head includes T.


In [ ]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, gradient):
        return -ctx.lambd * gradient, None

def mlp(in_dim, hidden, out_dim, dropout):
    layers = []
    for width in hidden:
        layers += [nn.Linear(in_dim, width), nn.ReLU(), nn.Dropout(dropout)]
        in_dim = width
    return nn.Sequential(*layers, nn.Linear(in_dim, out_dim))

class AdjustmentContinuous(nn.Module):
    def __init__(self, dH, dW, cfg):
        super().__init__()
        dZ, drop = cfg["dZ"], cfg["dropout"]
        self.encoder = mlp(dH, cfg["encoder_hidden"], dZ, drop)
        self.y_head = mlp(dZ + dW + 1, cfg["outcome_hidden"], 1, drop)
        self.t_adv = mlp(dZ, cfg["adversary_hidden"], 1, drop)

def tensor(x):
    return torch.as_tensor(x, dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def encode_batches(model, h, batch_size=2048):
    model.eval()
    return np.concatenate([
        model.encoder(tensor(h[i:i + batch_size])).cpu().numpy()
        for i in range(0, len(h), batch_size)
    ])

def fit_representation(htr, wtr, ttr, ytr, hva, wva, tva, yva, seed, cfg):
    set_seed(seed)
    model = AdjustmentContinuous(htr.shape[1], wtr.shape[1], cfg).to(DEVICE)
    main_params = list(model.encoder.parameters()) + list(model.y_head.parameters())
    adv_params = list(model.t_adv.parameters())
    main_opt = torch.optim.AdamW(main_params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    adv_opt = torch.optim.AdamW(adv_params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    train_data = TensorDataset(*[torch.tensor(a, dtype=torch.float32)
                                for a in [htr, wtr, ttr, ytr]])
    loader = DataLoader(train_data, batch_size=cfg["batch_size"],
                        shuffle=True, num_workers=0)
    hv, wv, tv, yv = [tensor(a) for a in [hva, wva, tva, yva]]
    best, best_state, best_epoch, bad = float("inf"), None, None, 0
    history = []
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        train_y_sum, train_adv_sum, n_train = 0.0, 0.0, 0
        for hb, wb, tb, yb in loader:
            hb, wb, tb, yb = [a.to(DEVICE) for a in [hb, wb, tb, yb]]
            # Main update: reverse the gradient going to Z, freeze adversary weights.
            for p in adv_params:
                p.requires_grad_(False)
            main_opt.zero_grad(set_to_none=True)
            z = model.encoder(hb)
            yhat = model.y_head(torch.cat([z, wb, tb[:, None]], dim=1)).squeeze(-1)
            that = model.t_adv(GradReverse.apply(z, cfg["grl_lambda"])).squeeze(-1)
            ly, la = F.mse_loss(yhat, yb), F.mse_loss(that, tb)
            (ly + cfg["adv_weight"] * la).backward()
            nn.utils.clip_grad_norm_(main_params, 5.0)
            main_opt.step()
            # Adversary update: no gradient to encoder.
            for p in adv_params:
                p.requires_grad_(True)
            adv_opt.zero_grad(set_to_none=True)
            with torch.no_grad():
                detached_z = model.encoder(hb)
            adv_loss = F.mse_loss(model.t_adv(detached_z).squeeze(-1), tb)
            adv_loss.backward()
            nn.utils.clip_grad_norm_(adv_params, 5.0)
            adv_opt.step()
            train_y_sum += ly.item() * len(hb)
            train_adv_sum += la.item() * len(hb)
            n_train += len(hb)
        model.eval()
        with torch.no_grad():
            zv = model.encoder(hv)
            yp = model.y_head(torch.cat([zv, wv, tv[:, None]], dim=1)).squeeze(-1)
            tp = model.t_adv(zv).squeeze(-1)
            val_y = F.mse_loss(yp, yv).item()
            val_adv = F.mse_loss(tp, tv).item()
        assert np.isfinite(val_y) and np.isfinite(val_adv), "Nonfinite training loss."
        history.append({"epoch": epoch, "train_y_mse": train_y_sum / n_train,
            "train_adv_mse": train_adv_sum / n_train,
            "val_y_mse": val_y, "val_adv_mse": val_adv})
        # Only checkpoints after the minimum warm-up are eligible.
        if epoch >= cfg["min_epochs"]:
            if val_y < best - 1e-6:
                best, best_epoch, bad = val_y, epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= cfg["patience"]:
                    break
    assert best_state is not None, "epochs must be >= min_epochs"
    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(history), best_epoch

def r2_np(actual, predicted):
    denom = np.sum((actual - np.mean(actual)) ** 2)
    return float(1 - np.sum((actual - predicted) ** 2) / denom) if denom > 1e-12 else np.nan

def make_learner(kind, seed):
    if kind == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=CFG["ridge_alpha"], solver="cholesky"))
    if kind == "RF":
        return RandomForestRegressor(
            n_estimators=CFG["rf_n_estimators"], max_depth=CFG["rf_max_depth"],
            min_samples_leaf=CFG["rf_min_samples_leaf"],
            random_state=seed, n_jobs=CPU_THREADS, max_features=CFG["rf_max_features"],
        )
    raise ValueError(kind)

def nuisance_predictions(xtr, xte, ttr, ytr, learner, seed):
    treatment_model = make_learner(learner, seed)
    outcome_model = make_learner(learner, seed + 1000)
    treatment_model.fit(xtr, ttr)
    outcome_model.fit(xtr, ytr)
    return treatment_model.predict(xte), outcome_model.predict(xte)


probe=torch.tensor([1.],requires_grad=True)
GradReverse.apply(probe,.05).sum().backward()
assert torch.allclose(probe.grad,torch.tensor([-.05]))


## Step 5 — Run / resume the Monte Carlo
Long-running step. The progress bar estimates remaining hours from completed **new** tasks on your hardware; no untested wall-clock promise is made. Each task fits all ten methods. After each complete outer fold its predictions/probes are atomically saved; completed tasks are reused. If interrupted, restart the kernel and rerun Steps 1–5 with unchanged settings. Only an interrupted fold retrains. One writer per output folder.

Primary sample resampling is across distinct companies, followed by one randomly selected annual report per company. This produces repeated draws from a fixed finite real-design population and fresh independent T/Y errors. Intervals are score-based approximate intervals; coverage is an empirical test of the entire learned pipeline under this DGP, not an assumed consequence of cross-fitting. No pooling of Z coordinates across different encoders is used in probes.


In [ ]:
METHODS=[f'{view}-{kind}' for view in ['W_only','Raw_H','NoAdv','Adjustment'] for kind in ['Ridge','RF']]+['Oracle_components-Ridge','Oracle_exact']
PROBE_TARGETS=['C','M','L','T','g_Y_true']

def fit_w_sim(wn,wc,tr):
 return StandardScaler().fit(wn[tr]),OneHotEncoder(handle_unknown='ignore',sparse_output=False,dtype=np.float32).fit(wc[tr])
def apply_w_sim(prep,wn,wc,rows):
 return np.column_stack([prep[0].transform(wn[rows]),prep[1].transform(wc[rows])]).astype(np.float32)

def make_folds(groups,seed):
 ordered=np.random.default_rng(seed).permutation(np.unique(groups));mapping={g:i for i,g in enumerate(ordered)}
 folds=list(GroupKFold(n_splits=CFG['n_splits']).split(groups,groups=np.array([mapping[g] for g in groups])))
 for tr,te in folds:assert set(groups[tr]).isdisjoint(groups[te])
 return folds

def neural_view(h,wn,wc,t,y,groups,tr,te,seed,view):
 inner=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=seed)
 i,j=next(inner.split(h[tr],groups=groups[tr]));itr,iva=tr[i],tr[j]
 assert set(groups[itr]).isdisjoint(groups[iva])
 hs=StandardScaler().fit(h[itr]);wp=fit_w_sim(wn,wc,itr)
 ts=StandardScaler().fit(t[itr,None]);ys=StandardScaler().fit(y[itr,None])
 cfg=copy.deepcopy(CFG)
 if view=='NoAdv':cfg.update(adv_weight=0.,grl_lambda=0.)
 model,history,epoch=fit_representation(
  hs.transform(h[itr]),apply_w_sim(wp,wn,wc,itr),ts.transform(t[itr,None]).ravel(),ys.transform(y[itr,None]).ravel(),
  hs.transform(h[iva]),apply_w_sim(wp,wn,wc,iva),ts.transform(t[iva,None]).ravel(),ys.transform(y[iva,None]).ravel(),seed,cfg)
 zsc=StandardScaler().fit(encode_batches(model,hs.transform(h[itr])))
 ztr=zsc.transform(encode_batches(model,hs.transform(h[tr]))).astype(np.float32)
 zte=zsc.transform(encode_batches(model,hs.transform(h[te]))).astype(np.float32)
 del model;gc.collect()
 return ztr,zte,history,epoch

def run_task(rows,factors,groups,scenario_name,params,tau,seed,rep,task_dir):
 complete_path=task_dir/'complete.json'
 if complete_path.is_file():
  receipt=json.loads(complete_path.read_text())
  assert receipt['complete'] and receipt['methods']==METHODS
  result=pd.read_csv(task_dir/'estimates.csv');diag=pd.read_csv(task_dir/'diagnostics.csv')
  assert len(result)==len(METHODS) and set(result.method)==set(METHODS)
  return result,diag,True
 task_dir.mkdir(parents=True,exist_ok=True)
 d=generate_dgp(factors,params,tau,seed);t,y=d['T'],d['Y']
 h=source_H[rows];wn=factors['W_numeric'];wc=source.iloc[rows][CAT_COLS].astype(str).to_numpy()
 folds=make_folds(groups,seed)
 pred={method:np.full((len(rows),2),np.nan) for method in METHODS}
 true_probe=np.column_stack([factors[k] for k in ['C','M','L']]+[t,d['g_true']])
 probe_pred={view:np.full_like(true_probe,np.nan) for view in ['Raw_H','NoAdv','Adjustment']}
 atomic_npz(task_dir/'generated_data.npz',source_rows=rows,T=t,Y=y,m_true=d['m_true'],ell_true=d['ell_true'],C=factors['C'],M=factors['M'],L=factors['L'])
 for fold,(tr,te) in enumerate(folds,1):
  path=task_dir/f'fold_{fold}.npz'
  if path.is_file():
   with np.load(path,allow_pickle=False) as s:
    assert np.array_equal(s['train_idx'],tr) and np.array_equal(s['test_idx'],te)
    for method in METHODS:pred[method][te]=s['pred_'+method]
    for view in probe_pred:probe_pred[view][te]=s['probe_'+view]
   continue
  wp=fit_w_sim(wn,wc,tr);wtr=apply_w_sim(wp,wn,wc,tr);wte=apply_w_sim(wp,wn,wc,te)
  hs=StandardScaler().fit(h[tr]);htr=hs.transform(h[tr]);hte=hs.transform(h[te])
  views={'W_only':(wtr,wte),'Raw_H':(np.column_stack([htr,wtr]),np.column_stack([hte,wte]))}
  for view in ['NoAdv','Adjustment']:
   ztr,zte,history,epoch=neural_view(h,wn,wc,t,y,groups,tr,te,seed+fold,view)
   views[view]=(np.column_stack([ztr,wtr]),np.column_stack([zte,wte]))
   history.assign(selected_epoch=epoch).to_csv(task_dir/f'{view}_history_fold{fold}.csv',index=False)
  for view in ['W_only','Raw_H','NoAdv','Adjustment']:
   for kind in ['Ridge','RF']:
    method=f'{view}-{kind}'
    tp,yp=nuisance_predictions(*views[view],t[tr],y[tr],kind,seed+fold)
    pred[method][te]=np.column_stack([tp,yp])
  oracle_x=np.column_stack([factors[k] for k in ['C','M','L','W_score']])
  tp,yp=nuisance_predictions(oracle_x[tr],oracle_x[te],t[tr],y[tr],'Ridge',seed+fold)
  pred['Oracle_components-Ridge'][te]=np.column_stack([tp,yp])
  pred['Oracle_exact'][te]=np.column_stack([d['m_true'][te],d['ell_true'][te]])
  # Same held-out multivariate Ridge probe across representations; truth used only for evaluation.
  for view in probe_pred:
   diagnostic=make_learner('Ridge',seed+fold)
   diagnostic.fit(views[view][0],true_probe[tr])
   probe_pred[view][te]=diagnostic.predict(views[view][1])
  save={'train_idx':tr,'test_idx':te}
  save.update({'pred_'+m:pred[m][te] for m in METHODS})
  save.update({'probe_'+v:probe_pred[v][te] for v in probe_pred})
  atomic_npz(path,**save)
  if torch.cuda.is_available():torch.cuda.empty_cache()
 results=[];diagnostics=[]
 for method in METHODS:
  tp,yp=pred[method].T
  assert np.isfinite(tp).all() and np.isfinite(yp).all()
  results.append({'scenario':scenario_name,'replication':rep,'seed':seed,'true_tau':tau,'method':method,
                  **score_summary(t,y,tp,yp,groups,tau)})
 for view in probe_pred:
  for k,name in enumerate(PROBE_TARGETS):
   diagnostics.append({'scenario':scenario_name,'replication':rep,'true_tau':tau,'view':view,
                       'target':name,'R2_oof':r2_np(true_probe[:,k],probe_pred[view][:,k])})
 result=pd.DataFrame(results);diag=pd.DataFrame(diagnostics)
 result.to_csv(task_dir/'estimates.csv',index=False);diag.to_csv(task_dir/'diagnostics.csv',index=False)
 cal={'scenario':scenario_name,'replication':rep,'true_tau':tau,'gamma_c':params['gamma_c'],'gamma_l':params['gamma_l'],
      'corr_T_C':float(np.corrcoef(t,factors['C'])[0,1]),'corr_T_L':float(np.corrcoef(t,factors['L'])[0,1]),
      'T_sd':float(t.std()),'Y_sd':float(y.std()),'known_assignment_noise_variance':SIM['sigma_t']**2}
 atom_json(task_dir/'calibration.json',cal)
 atom_json(complete_path,{'complete':True,'methods':METHODS,'reports':len(rows)})
 return result,diag,False

def run_monte_carlo():
 total=SIM['replications']*len(SCENARIOS)*len(SIM['true_taus']);new_times=[]
 result_parts=[];diag_parts=[];cal_parts=[]
 with tqdm(total=total,desc='Semi-synthetic tasks (each = all 10 methods)') as progress:
  for rep in range(SIM['replications']):
   seed=SIM['base_seed']+rep
   candidate=one_report_per_company(evaluation_frame,seed)
   chosen=np.random.default_rng(seed).choice(candidate,SIM['n_companies'],replace=False)
   rows=evaluation_rows[chosen];groups=source.iloc[rows].gvkey.astype(str).to_numpy()
   assert len(np.unique(groups))==len(rows)
   factors={k:v[rows] for k,v in all_factors.items()}
   source.iloc[rows][['doc_id','gvkey']].to_csv(RUN_DIR/f'sample_rep_{rep:03d}.csv',index=False)
   for tau in SIM['true_taus']:
    for scenario,params in SCENARIOS.items():
     task=RUN_DIR/'tasks'/f'rep_{rep:03d}_tau_{tau:g}_{scenario}'
     begin=time.perf_counter()
     result,diag,reused=run_task(rows,factors,groups,scenario,params,tau,seed,rep,task)
     if not reused:new_times.append(time.perf_counter()-begin)
     result_parts.append(result);diag_parts.append(diag)
     cal_parts.append(json.loads((task/'calibration.json').read_text()))
     # Completed task files are authoritative recovery checkpoints.
     pd.concat(result_parts,ignore_index=True).to_csv(RUN_DIR/'mc_results_so_far.csv',index=False)
     progress.update(1)
     if new_times:
      remaining=total-progress.n
      progress.set_postfix(task_seconds=round(new_times[-1],1),projected_remaining_hours=round(np.mean(new_times)*remaining/3600,2))
 all_results=pd.concat(result_parts,ignore_index=True);all_diagnostics=pd.concat(diag_parts,ignore_index=True)
 assert len(all_results)==total*len(METHODS)
 assert not all_results.duplicated(['scenario','replication','true_tau','method']).any()
 all_results.to_csv(RUN_DIR/'monte_carlo_raw.csv',index=False)
 all_diagnostics.to_csv(RUN_DIR/'representation_recovery_raw.csv',index=False)
 pd.DataFrame(cal_parts).to_csv(RUN_DIR/'dgp_calibration.csv',index=False)
 atom_json(RUN_DIR/'monte_carlo_complete.json',{'complete':True,'tasks':total,'methods':METHODS,'replications_per_scenario':SIM['replications']})
 return all_results,all_diagnostics

started=time.perf_counter()
mc_results,mc_diagnostics=run_monte_carlo()
print('Elapsed hours:',round((time.perf_counter()-started)/3600,3))
print('All raw results:',RUN_DIR)


## Step 6 — Evaluate known-truth accuracy and export plots
Bias = average signed estimation error. Absolute bias = absolute value of that average. MAE = mean absolute replication error; these are different statistics. Report RMSE, empirical SD, mean reported SE, 95% interval coverage, Monte Carlo SEs and held-out signal-recovery probes. The primary ablation is Adjustment versus NoAdv with the **same nuisance learner**.

Paired gains are baseline loss minus Adjustment loss; positive values favor Adjustment. Their intervals quantify Monte Carlo uncertainty, not an empirical ROA causal effect. Comparisons across multiple scenarios are exploratory; do not select or hide scenarios based on which method wins. Fifty repetitions are a first test (coverage changes in increments of 2 percentage points); increase repetitions only after the design and implementation are settled. Optional `true_taus=[0.0,1.0]` adds a null-effect check and doubles runtime.


In [ ]:
def summarize_mc(results):
 rows=[]
 for (scenario,tau,method),part in results.groupby(['scenario','true_tau','method'],sort=False):
  err=part.error.to_numpy();r=len(err);coverage=part.covered.astype(float).mean()
  bias=err.mean();sd=part.estimate.std(ddof=1);bias_mcse=err.std(ddof=1)/np.sqrt(r)
  crit=student_t.ppf(.975,r-1)
  rows.append({'scenario':scenario,'true_tau':tau,'method':method,'replications':r,
   'bias':bias,'absolute_bias':abs(bias),'bias_mcse':bias_mcse,
   'bias_mc_low':bias-crit*bias_mcse,'bias_mc_high':bias+crit*bias_mcse,
   'MAE':np.mean(abs(err)),'RMSE':np.sqrt(np.mean(err**2)),'empirical_sd':sd,
   'mean_reported_se':part.se.mean(),'coverage_95':coverage,
   'coverage_mcse':np.sqrt(coverage*(1-coverage)/r),
   'mean_ci_width':(part.ci_high-part.ci_low).mean(),'mean_R2_T':part.R2_T.mean(),
   'mean_R2_Y':part.R2_Y.mean(),'mean_residual_variance_ratio':part.residual_mse_to_T_variance.mean()})
 return pd.DataFrame(rows)

def paired_comparisons(results):
 rows=[]
 for (scenario,tau),part in results.groupby(['scenario','true_tau'],sort=False):
  for learner in ['Ridge','RF']:
   adjustment='Adjustment-'+learner
   for baseline in ['NoAdv-'+learner,'Raw_H-'+learner,'W_only-'+learner]:
    pair=part.loc[part.method.isin([adjustment,baseline])].pivot(index='replication',columns='method',values=['absolute_error','squared_error'])
    assert not pair.isna().any().any()
    for metric in ['absolute_error','squared_error']:
     # Positive gain means lower error for Adjustment. These are Monte Carlo paired comparisons.
     gain=(pair[metric][baseline]-pair[metric][adjustment]).to_numpy();r=len(gain)
     mean=gain.mean();se=gain.std(ddof=1)/np.sqrt(r);crit=student_t.ppf(.975,r-1)
     base_mean=pair[metric][baseline].mean()
     rows.append({'scenario':scenario,'true_tau':tau,'Adjustment':adjustment,'baseline':baseline,'loss':metric,
      'paired_replications':r,'mean_gain':mean,'gain_mcse':se,'gain_ci_low':mean-crit*se,
      'gain_ci_high':mean+crit*se,'Adjustment_win_share':np.mean(gain>0),
      'loss_reduction_pct':100*mean/base_mean if base_mean>1e-12 else np.nan})
 return pd.DataFrame(rows)

summary=summarize_mc(mc_results)
paired=paired_comparisons(mc_results)
recovery=mc_diagnostics.groupby(['scenario','true_tau','view','target'],sort=False).R2_oof.agg(['mean','std','count']).reset_index()
recovery['mcse']=recovery['std']/np.sqrt(recovery['count'])
summary.to_csv(RUN_DIR/'simulation_summary.csv',index=False)
paired.to_csv(RUN_DIR/'paired_Adjustment_comparisons.csv',index=False)
recovery.to_csv(RUN_DIR/'signal_recovery_summary.csv',index=False)
print('Bias = mean(estimate - truth); MAE is not the same quantity as absolute bias.')
display(summary[['scenario','method','bias','bias_mcse','MAE','RMSE','coverage_95','mean_reported_se']].round(4))
print('Primary same-budget ablation: positive gain favors Adjustment; negative gain favors NoAdv.')
display(paired.loc[paired.baseline.str.startswith('NoAdv')].round(4))
oracle=summary.loc[summary.method.eq('Oracle_exact')]
if (oracle.bias.abs()>3*oracle.bias_mcse).any():
 print('REVIEW: exact-oracle bias exceeds 3 Monte Carlo SE in at least one scenario. Check DGP/score and consider sampling fluctuation; do not tune Adjustment to fix this.')
print('At 50 replications, coverage is coarse (2 percentage points per replication).')

for tau in SIM['true_taus']:
 s=summary.loc[summary.true_tau.eq(tau)]
 fig,axes=plt.subplots(2,2,figsize=(14,10),squeeze=False)
 for ax,scenario in zip(axes.flat,SCENARIOS):
  d=s.loc[s.scenario.eq(scenario)].set_index('method').reindex(METHODS)
  yy=np.arange(len(d));ax.errorbar(d.bias,yy,xerr=[d.bias-d.bias_mc_low,d.bias_mc_high-d.bias],fmt='o',capsize=3)
  ax.axvline(0,color='black',linestyle='--');ax.set_yticks(yy);ax.set_yticklabels(d.index)
  ax.set_title(scenario);ax.set_xlabel('Monte Carlo bias (95% Monte Carlo interval)');ax.invert_yaxis()
 fig.suptitle(f'True constant slope = {tau:g}; 50-replication default')
 fig.tight_layout();fig.savefig(RUN_DIR/f'bias_tau_{tau:g}.png',dpi=180);plt.show();plt.close(fig)
 fig,axes=plt.subplots(1,2,figsize=(14,6))
 for ax,metric,title in zip(axes,['RMSE','coverage_95'],['Estimation RMSE','Empirical 95% interval coverage']):
  mat=s.pivot(index='method',columns='scenario',values=metric).reindex(index=METHODS,columns=list(SCENARIOS))
  image=ax.imshow(mat.to_numpy(),aspect='auto',cmap='viridis',vmin=0,vmax=1 if metric=='coverage_95' else None)
  ax.set_xticks(range(len(mat.columns)));ax.set_xticklabels(mat.columns,rotation=35,ha='right')
  ax.set_yticks(range(len(mat)));ax.set_yticklabels(mat.index);ax.set_title(title)
  for i in range(len(mat)):
   for j in range(len(mat.columns)):ax.text(j,i,f'{mat.iloc[i,j]:.3f}',ha='center',va='center',color='white',fontsize=8)
  fig.colorbar(image,ax=ax)
 fig.tight_layout();fig.savefig(RUN_DIR/f'rmse_coverage_tau_{tau:g}.png',dpi=180);plt.show();plt.close(fig)
 fig,axes=plt.subplots(2,2,figsize=(12,8))
 for ax,target in zip(axes.flat,['C','L','M','T']):
  for view in ['Raw_H','NoAdv','Adjustment']:
   d=recovery.loc[recovery.true_tau.eq(tau)&recovery.target.eq(target)&recovery['view'].eq(view)].set_index('scenario').reindex(SCENARIOS)
   ax.errorbar(range(len(SCENARIOS)),d['mean'],yerr=1.96*d.mcse,marker='o',label=view,capsize=3)
  ax.set_xticks(range(len(SCENARIOS)));ax.set_xticklabels(SCENARIOS,rotation=25,ha='right',fontsize=8)
  ax.set_title(f'Held-out prediction of {target} from adjustment features (including W)')
  ax.set_ylabel('OOF Ridge R2');ax.legend()
 fig.tight_layout();fig.savefig(RUN_DIR/f'signal_recovery_tau_{tau:g}.png',dpi=180);plt.show();plt.close(fig)
print('Send back: simulation_summary.csv, paired_Adjustment_comparisons.csv, signal_recovery_summary.csv, dgp_calibration.csv and figures.')
print('Output folder:',RUN_DIR)


## What this simulation can and cannot establish
A method with lower bias/MAE/RMSE and well-calibrated coverage, retaining C and M under the fixed scenarios, is better supported **under these data-generating assumptions**. Adjustment may fail when adversarial removal deletes real confounders, when training is too weak, or when Raw H already provides valid sufficient adjustment. Report these failures. Success does not establish that the real competition–ROA effect is identified, nor does it resolve actual filing timing or historical identity.

Send back `simulation_summary.csv`, `paired_Adjustment_comparisons.csv`, `signal_recovery_summary.csv`, `dgp_calibration.csv`, and the generated PNG figures. Retain configuration and task folders for recovery. No empirical dataset is overwritten.
